# Camera A Heat-Impulse Phase Correlation Analysis

**Core task:** Compare motion of single tracked spot (target ROI) vs. whole detector plane (binned field) under heat impulse.

- Load baseline and test frames from FITS
- Phase correlate target ROI → measure displacement in pixels
- Phase correlate binned whole field → measure displacement in pixels
- Analyse difference and generate visualizations

## 1. Setup and Configuration

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from astropy.io import fits
from scipy.ndimage import gaussian_filter
from skimage.registration import phase_cross_correlation
from pathlib import Path
from PIL import Image
import io
import pandas as pd

# ============ USER SETTINGS ============
DATA_DIR = Path(r"C:\Users\biswa\Heloo exoplanets")

# FITS files to analyse
BASELINE_FILE = "cameraA_frame_0001_baseline.fits"
TEST_FILES = [
    "cameraA_frame_0024_pulse_midpoint.fits",
    "cameraA_frame_0055_max_response.fits",
]

# Target spot (bright feature to track)
TARGET_X_PX = 3963.637
TARGET_Y_PX = 2981.505
TARGET_ROI_SIZE = 160  # 160×160 px window around target

# Whole-field analysis
FIELD_BIN_FACTOR = 8  # Bin full frame 8× to make phase corr tractable

# Phase correlation settings
TARGET_UPSAMPLE = 100  # Sub-pixel accuracy for target
FIELD_UPSAMPLE = 30    # Sub-pixel accuracy for field
BLUR_SIGMA = 15.0      # Gaussian blur for high-pass preprocessing

# Output
OUTPUT_DIR = DATA_DIR / "analysis_output"
OUTPUT_DIR.mkdir(exist_ok=True)

print(f"Data dir: {DATA_DIR}")
print(f"Output dir: {OUTPUT_DIR}")
print(f"Target ROI: ({TARGET_X_PX}, {TARGET_Y_PX}), size {TARGET_ROI_SIZE} px")
print(f"Field bin factor: {FIELD_BIN_FACTOR}×")

## 2. Helper Functions

In [ ]:
def load_fits(path):
    """Load 2D image from FITS file."""
    with fits.open(path) as hdul:
        for hdu in hdul:
            if hdu.data is not None and hdu.data.ndim >= 2:
                img = np.array(hdu.data, dtype=np.float32)
                if img.ndim > 2:
                    img = img[0]
                return img
    raise ValueError(f"No 2D image in {path}")

def crop_roi(image, cx_px, cy_px, size_px):
    """Crop square ROI centered on (cx_px, cy_px)."""
    half = size_px // 2
    x0 = int(np.round(cx_px)) - half
    y0 = int(np.round(cy_px)) - half
    x1, y1 = x0 + size_px, y0 + size_px
    
    if x0 < 0 or y0 < 0 or x1 > image.shape[1] or y1 > image.shape[0]:
        raise ValueError(f"ROI bounds ({x0}:{x1}, {y0}:{y1}) out of image ({image.shape})")
    return image[y0:y1, x0:x1].copy(), (x0, y0)

def bin_image(image, factor):
    """Bin image by mean, preserving physical size relationship."""
    factor = int(factor)
    if factor <= 1:
        return image.copy()
    ny, nx = image.shape
    ny_trim = (ny // factor) * factor
    nx_trim = (nx // factor) * factor
    trimmed = image[:ny_trim, :nx_trim]
    return trimmed.reshape(ny_trim // factor, factor, nx_trim // factor, factor).mean(axis=(1, 3))

def preprocess_for_correlation(image, blur_sigma=15.0):
    """High-pass + robust normalisation for phase correlation."""
    arr = np.asarray(image, dtype=np.float64)
    
    # High-pass: remove low spatial freq
    blurred = gaussian_filter(arr, sigma=blur_sigma)
    hp = arr - blurred
    
    # Robust standardise (median/MAD)
    med = np.median(hp)
    mad = np.median(np.abs(hp - med))
    scale = max(1.4826 * mad, 1e-10)
    hp_norm = (hp - med) / scale
    hp_norm = np.clip(hp_norm, -8, 8)
    
    # Taper edges to suppress Fourier wraparound
    window = np.outer(np.hanning(hp_norm.shape[0]), np.hanning(hp_norm.shape[1]))
    return hp_norm * window

def measure_motion(baseline, test, upsample=100):
    """
    Phase correlation: measure (dx, dy) shift from baseline to test.
    Returns: (dx_px, dy_px, error_px, ncc_before, ncc_after)
    """
    base_prep = preprocess_for_correlation(baseline, BLUR_SIGMA)
    test_prep = preprocess_for_correlation(test, BLUR_SIGMA)
    
    # Phase correlation
    shift, error, phasediff = phase_cross_correlation(
        base_prep, test_prep, upsample_factor=upsample, return_error=True
    )
    # shift = (dy, dx) from scikit-image convention
    dy, dx = shift
    
    # NCC before and after for quality control
    ncc_before = np.mean(base_prep * test_prep)
    
    # Shift test to alignment and re-correlate
    from scipy.ndimage import shift as ndi_shift
    test_shifted = ndi_shift(test_prep, (dy, dx))
    ncc_after = np.mean(base_prep * test_shifted)
    
    return dx, dy, float(error), float(ncc_before), float(ncc_after)

print("Helpers loaded.")

## 3. Load and Analyse

In [ ]:
# Load baseline
baseline_path = DATA_DIR / BASELINE_FILE
baseline_full = load_fits(baseline_path)
print(f"Baseline shape: {baseline_full.shape}")

# Extract baseline target ROI and field
baseline_target, target_origin = crop_roi(baseline_full, TARGET_X_PX, TARGET_Y_PX, TARGET_ROI_SIZE)
baseline_field = bin_image(baseline_full, FIELD_BIN_FACTOR)
baseline_field_fullres = baseline_full.copy()  # Full-res baseline for comparison

print(f"Baseline target ROI shape: {baseline_target.shape}")
print(f"Baseline field (binned {FIELD_BIN_FACTOR}×) shape: {baseline_field.shape}")
print(f"Baseline field (full-res) shape: {baseline_field_fullres.shape}")

# Analyse each test frame
results = []

for test_file in TEST_FILES:
    test_path = DATA_DIR / test_file
    test_full = load_fits(test_path)
    print(f"\n=== {test_file} ===")
    
    # ===== TARGET ROI ANALYSIS =====
    test_target, _ = crop_roi(test_full, TARGET_X_PX, TARGET_Y_PX, TARGET_ROI_SIZE)
    dx_target, dy_target, err_target, ncc_b_t, ncc_a_t = measure_motion(
        baseline_target, test_target, upsample=TARGET_UPSAMPLE
    )
    print(f"[TARGET ROI] Δx={dx_target:+.3f} px, Δy={dy_target:+.3f} px, error={err_target:.4f}")
    print(f"  NCC: before={ncc_b_t:.4f}, after={ncc_a_t:.4f}, gain={ncc_a_t - ncc_b_t:+.4f}")
    
    # ===== WHOLE FIELD (BINNED) ANALYSIS =====
    test_field = bin_image(test_full, FIELD_BIN_FACTOR)
    dx_field, dy_field, err_field, ncc_b_f, ncc_a_f = measure_motion(
        baseline_field, test_field, upsample=FIELD_UPSAMPLE
    )
    dx_field_native = dx_field * FIELD_BIN_FACTOR
    dy_field_native = dy_field * FIELD_BIN_FACTOR
    print(f"[WHOLE FIELD BINNED] Δx={dx_field_native:+.3f} px, Δy={dy_field_native:+.3f} px, error={err_field:.4f}")
    print(f"  NCC: before={ncc_b_f:.4f}, after={ncc_a_f:.4f}, gain={ncc_a_f - ncc_b_f:+.4f}")
    
    # ===== WHOLE FIELD (FULL-RES) ANALYSIS =====
    test_field_fullres = test_full.copy()
    dx_field_fr, dy_field_fr, err_field_fr, ncc_b_fr, ncc_a_fr = measure_motion(
        baseline_field_fullres, test_field_fullres, upsample=FIELD_UPSAMPLE
    )
    print(f"[WHOLE FIELD FULL-RES] Δx={dx_field_fr:+.3f} px, Δy={dy_field_fr:+.3f} px, error={err_field_fr:.4f}")
    print(f"  NCC: before={ncc_b_fr:.4f}, after={ncc_a_fr:.4f}, gain={ncc_a_fr - ncc_b_fr:+.4f}")
    
    # ===== DIFFERENCES =====
    dx_diff_binned = dx_target - dx_field_native
    dy_diff_binned = dy_target - dy_field_native
    dx_diff_fullres = dx_target - dx_field_fr
    dy_diff_fullres = dy_target - dy_field_fr
    
    print(f"[DIFF: Target - Binned Field] Δx={dx_diff_binned:+.3f} px, Δy={dy_diff_binned:+.3f} px")
    print(f"[DIFF: Target - Full-Res Field] Δx={dx_diff_fullres:+.3f} px, Δy={dy_diff_fullres:+.3f} px")
    print(f"[CONSISTENCY] Binned vs Full-Res field difference: Δx={dx_field_native - dx_field_fr:+.3f} px, Δy={dy_field_native - dy_field_fr:+.3f} px")
    
    results.append({
        'frame': test_file,
        # Target
        'target_dx': dx_target,
        'target_dy': dy_target,
        'target_err': err_target,
        'target_ncc_gain': ncc_a_t - ncc_b_t,
        # Field binned
        'field_binned_dx': dx_field_native,
        'field_binned_dy': dy_field_native,
        'field_binned_err': err_field,
        'field_binned_ncc_gain': ncc_a_f - ncc_b_f,
        # Field full-res
        'field_fullres_dx': dx_field_fr,
        'field_fullres_dy': dy_field_fr,
        'field_fullres_err': err_field_fr,
        'field_fullres_ncc_gain': ncc_a_fr - ncc_b_fr,
        # Differences
        'diff_target_vs_binned_dx': dx_diff_binned,
        'diff_target_vs_binned_dy': dy_diff_binned,
        'diff_target_vs_fullres_dx': dx_diff_fullres,
        'diff_target_vs_fullres_dy': dy_diff_fullres,
        'consistency_binned_vs_fullres_dx': dx_field_native - dx_field_fr,
        'consistency_binned_vs_fullres_dy': dy_field_native - dy_field_fr,
    })

results_df = pd.DataFrame(results)
print("\n" + "="*100)
print("RESULTS TABLE")
print("="*100)
print(results_df.to_string(index=False))

## 4. Summary Plot: Target vs. Field Motion

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 11))

frames_labels = [f.split('frame_')[1].split('.')[0] for f in results_df['frame']]
x_pos = np.arange(len(results_df))
width = 0.25

# ===== Row 1: X displacement =====
ax = axes[0, 0]
ax.bar(x_pos - width, results_df['target_dx'], width, label='Target ROI', alpha=0.85)
ax.bar(x_pos, results_df['field_binned_dx'], width, label='Field (binned)', alpha=0.85)
ax.bar(x_pos + width, results_df['field_fullres_dx'], width, label='Field (full-res)', alpha=0.85)
ax.axhline(0, color='k', linestyle='--', alpha=0.3, lw=1)
ax.set_ylabel('X displacement (px)', fontsize=11)
ax.set_xlabel('Frame', fontsize=11)
ax.set_title('X-axis Motion: Target vs. Field (Binned & Full-Res)', fontsize=12, fontweight='bold')
ax.set_xticks(x_pos)
ax.set_xticklabels(frames_labels, rotation=45, ha='right', fontsize=9)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3, axis='y')

# ===== Row 1: Y displacement =====
ax = axes[0, 1]
ax.bar(x_pos - width, results_df['target_dy'], width, label='Target ROI', alpha=0.85)
ax.bar(x_pos, results_df['field_binned_dy'], width, label='Field (binned)', alpha=0.85)
ax.bar(x_pos + width, results_df['field_fullres_dy'], width, label='Field (full-res)', alpha=0.85)
ax.axhline(0, color='k', linestyle='--', alpha=0.3, lw=1)
ax.set_ylabel('Y displacement (px)', fontsize=11)
ax.set_xlabel('Frame', fontsize=11)
ax.set_title('Y-axis Motion: Target vs. Field (Binned & Full-Res)', fontsize=12, fontweight='bold')
ax.set_xticks(x_pos)
ax.set_xticklabels(frames_labels, rotation=45, ha='right', fontsize=9)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3, axis='y')

# ===== Row 2: Target excess vs. Binned Field =====
ax = axes[1, 0]
colors_x = ['C2' if x > 0 else 'C3' for x in results_df['diff_target_vs_binned_dx']]
ax.bar(x_pos, results_df['diff_target_vs_binned_dx'], color=colors_x, alpha=0.7, edgecolor='k', linewidth=1.5)
ax.axhline(0, color='k', linestyle='-', alpha=1, lw=2)
ax.set_ylabel('Target − Field (binned) [px]', fontsize=11)
ax.set_xlabel('Frame', fontsize=11)
ax.set_title('Target Excess Motion (X): vs. Binned Field', fontsize=12, fontweight='bold')
ax.set_xticks(x_pos)
ax.set_xticklabels(frames_labels, rotation=45, ha='right', fontsize=9)
ax.grid(True, alpha=0.3, axis='y')

# ===== Row 2: Target excess vs. Full-Res Field =====
ax = axes[1, 1]
colors_y = ['C2' if y > 0 else 'C3' for y in results_df['diff_target_vs_fullres_dy']]
ax.bar(x_pos, results_df['diff_target_vs_fullres_dy'], color=colors_y, alpha=0.7, edgecolor='k', linewidth=1.5)
ax.axhline(0, color='k', linestyle='-', alpha=1, lw=2)
ax.set_ylabel('Target − Field (full-res) [px]', fontsize=11)
ax.set_xlabel('Frame', fontsize=11)
ax.set_title('Target Excess Motion (Y): vs. Full-Res Field', fontsize=12, fontweight='bold')
ax.set_xticks(x_pos)
ax.set_xticklabels(frames_labels, rotation=45, ha='right', fontsize=9)
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "01_motion_comparison_all_three.png", dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved: {OUTPUT_DIR / '01_motion_comparison_all_three.png'}")

# Print consistency check
print("\n" + "="*70)
print("BINNED vs FULL-RES FIELD CONSISTENCY CHECK")
print("="*70)
for idx, (_, row) in enumerate(results_df.iterrows()):
    dx_diff = row['consistency_binned_vs_fullres_dx']
    dy_diff = row['consistency_binned_vs_fullres_dy']
    magnitude = np.sqrt(dx_diff**2 + dy_diff**2)
    print(f"{row['frame']}: Δx={dx_diff:+.3f} px, Δy={dy_diff:+.3f} px, magnitude={magnitude:.3f} px")

## 5. Difference Plot: Target Excess Motion

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

frames_labels = [f.split('frame_')[1].split('.')[0] for f in results_df['frame']]
x_pos = np.arange(len(results_df))
width = 0.35

# ===== Panel 1: X difference (Target vs Binned) =====
ax = axes[0, 0]
colors_x = ['C2' if x > 0 else 'C3' for x in results_df['diff_target_vs_binned_dx']]
ax.bar(x_pos, results_df['diff_target_vs_binned_dx'], color=colors_x, alpha=0.7, edgecolor='k', linewidth=1.5)
ax.axhline(0, color='k', linestyle='-', alpha=1, lw=2)
ax.set_ylabel('Δx (target) − Δx (field) [px]', fontsize=11)
ax.set_xlabel('Frame', fontsize=11)
ax.set_title('Target X-Excess (vs. Binned Field)', fontsize=12, fontweight='bold')
ax.set_xticks(x_pos)
ax.set_xticklabels(frames_labels, rotation=45, ha='right', fontsize=9)
ax.grid(True, alpha=0.3, axis='y')

# ===== Panel 2: Y difference (Target vs Binned) =====
ax = axes[0, 1]
colors_y = ['C2' if y > 0 else 'C3' for y in results_df['diff_target_vs_binned_dy']]
ax.bar(x_pos, results_df['diff_target_vs_binned_dy'], color=colors_y, alpha=0.7, edgecolor='k', linewidth=1.5)
ax.axhline(0, color='k', linestyle='-', alpha=1, lw=2)
ax.set_ylabel('Δy (target) − Δy (field) [px]', fontsize=11)
ax.set_xlabel('Frame', fontsize=11)
ax.set_title('Target Y-Excess (vs. Binned Field)', fontsize=12, fontweight='bold')
ax.set_xticks(x_pos)
ax.set_xticklabels(frames_labels, rotation=45, ha='right', fontsize=9)
ax.grid(True, alpha=0.3, axis='y')

# ===== Panel 3: X difference (Target vs Full-Res) =====
ax = axes[1, 0]
colors_x_fr = ['C2' if x > 0 else 'C3' for x in results_df['diff_target_vs_fullres_dx']]
ax.bar(x_pos, results_df['diff_target_vs_fullres_dx'], color=colors_x_fr, alpha=0.7, edgecolor='k', linewidth=1.5)
ax.axhline(0, color='k', linestyle='-', alpha=1, lw=2)
ax.set_ylabel('Δx (target) − Δx (field) [px]', fontsize=11)
ax.set_xlabel('Frame', fontsize=11)
ax.set_title('Target X-Excess (vs. Full-Res Field)', fontsize=12, fontweight='bold')
ax.set_xticks(x_pos)
ax.set_xticklabels(frames_labels, rotation=45, ha='right', fontsize=9)
ax.grid(True, alpha=0.3, axis='y')

# ===== Panel 4: Y difference (Target vs Full-Res) =====
ax = axes[1, 1]
colors_y_fr = ['C2' if y > 0 else 'C3' for y in results_df['diff_target_vs_fullres_dy']]
ax.bar(x_pos, results_df['diff_target_vs_fullres_dy'], color=colors_y_fr, alpha=0.7, edgecolor='k', linewidth=1.5)
ax.axhline(0, color='k', linestyle='-', alpha=1, lw=2)
ax.set_ylabel('Δy (target) − Δy (field) [px]', fontsize=11)
ax.set_xlabel('Frame', fontsize=11)
ax.set_title('Target Y-Excess (vs. Full-Res Field)', fontsize=12, fontweight='bold')
ax.set_xticks(x_pos)
ax.set_xticklabels(frames_labels, rotation=45, ha='right', fontsize=9)
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "02_target_excess_motion_all_three.png", dpi=150, bbox_inches='tight')
plt.show()
print(f"✓ Saved: {OUTPUT_DIR / '02_target_excess_motion_all_three.png'}")

## 6. ROI Visualisation: Baseline + Test Overlay

In [ ]:
# Create side-by-side comparison for each test frame
for idx, test_file in enumerate(TEST_FILES):
    test_path = DATA_DIR / test_file
    test_full = load_fits(test_path)
    test_target, _ = crop_roi(test_full, TARGET_X_PX, TARGET_Y_PX, TARGET_ROI_SIZE)
    
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    
    # Baseline target
    ax = axes[0]
    im = ax.imshow(baseline_target, cmap='gray', origin='lower')
    ax.set_title('Baseline Target ROI', fontsize=12, fontweight='bold')
    ax.set_xlabel('X (px)')
    ax.set_ylabel('Y (px)')
    plt.colorbar(im, ax=ax, label='ADU')
    
    # Test target
    ax = axes[1]
    im = ax.imshow(test_target, cmap='gray', origin='lower')
    row = results_df[results_df['frame'] == test_file].iloc[0]
    title_text = (
        f"Test: {test_file}\n"
        f"Δx={row['target_dx']:+.2f} px, Δy={row['target_dy']:+.2f} px"
    )
    ax.set_title(title_text, fontsize=12, fontweight='bold')
    ax.set_xlabel('X (px)')
    ax.set_ylabel('Y (px)')
    plt.colorbar(im, ax=ax, label='ADU')
    
    plt.tight_layout()
    outpath = OUTPUT_DIR / f"03_roi_baseline_vs_{idx+1}.png"
    plt.savefig(outpath, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"Saved: {outpath}")

## 7. Animation: Target ROI Blink (Baseline ↔ Test)

In [ ]:
from IPython.display import Image as IPImage, display
from base64 import b64encode

def make_blink_gif_inline(baseline_img, test_img, filename, frame_duration_ms=200, n_cycles=3):
    """Create alternating baseline-test animation and display inline."""
    frames = []
    for _ in range(n_cycles):
        frames.append(normalize_for_display(baseline_img))
        frames.append(normalize_for_display(test_img))
    
    pil_frames = []
    for frame in frames:
        uint8_frame = (frame * 255).astype(np.uint8)
        pil_img = Image.fromarray(uint8_frame, mode='L')
        pil_frames.append(pil_img)
    
    pil_frames[0].save(
        filename,
        save_all=True,
        append_images=pil_frames[1:],
        duration=frame_duration_ms,
        loop=0
    )
    print(f"✓ Saved: {filename}")
    
    # Display inline in notebook
    display(IPImage(filename=str(filename)))

# Create blink GIFs for each test frame
for idx, test_file in enumerate(TEST_FILES):
    print(f"\n--- Blink Animation {idx+1}: {test_file} ---")
    test_path = DATA_DIR / test_file
    test_full = load_fits(test_path)
    test_target, _ = crop_roi(test_full, TARGET_X_PX, TARGET_Y_PX, TARGET_ROI_SIZE)
    
    filename = OUTPUT_DIR / f"04_blink_roi_{idx+1}.gif"
    make_blink_gif_inline(baseline_target, test_target, filename, frame_duration_ms=250, n_cycles=4)

## 8. Animation: Motion Direction Arrows (Vector Field)

In [ ]:
def make_motion_vectors_gif_inline(baseline_full, test_full, results_row, output_file, bin_factor=16):
    """
    Create overlay showing target vs. field motion and display inline.
    """
    frames = []
    
    for img_label, img in [("Baseline", baseline_full), ("Test", test_full)]:
        fig, ax = plt.subplots(figsize=(11, 13))
        
        # Show downsampled image
        display_img = bin_image(img, bin_factor)
        display_img_norm = normalize_for_display(display_img)
        ax.imshow(display_img_norm, cmap='gray', origin='lower')
        
        # Plot target location (in binned coords)
        target_x_binned = TARGET_X_PX / bin_factor
        target_y_binned = TARGET_Y_PX / bin_factor
        ax.plot(target_x_binned, target_y_binned, 'r*', markersize=25, label='Target spot', zorder=5)
        
        if img_label == "Test":
            scale = 15  # Arrow length scale
            
            # TARGET MOTION (red)
            dx_t = results_row['target_dx'] / bin_factor
            dy_t = results_row['target_dy'] / bin_factor
            ax.arrow(
                target_x_binned, target_y_binned, dx_t * scale, dy_t * scale,
                head_width=35, head_length=25, fc='red', ec='darkred', linewidth=3.5,
                label=f'Target: ({results_row["target_dx"]:+.2f}, {results_row["target_dy"]:+.2f}) px',
                zorder=4
            )
            
            # FIELD MOTION - BINNED (blue)
            dx_fb = results_row['field_binned_dx'] / bin_factor
            dy_fb = results_row['field_binned_dy'] / bin_factor
            ax.arrow(
                target_x_binned + 120, target_y_binned, dx_fb * scale, dy_fb * scale,
                head_width=35, head_length=25, fc='blue', ec='darkblue', linewidth=3.5,
                label=f'Field (binned): ({results_row["field_binned_dx"]:+.2f}, {results_row["field_binned_dy"]:+.2f}) px',
                zorder=4
            )
            
            # FIELD MOTION - FULL-RES (green)
            dx_fr = results_row['field_fullres_dx'] / bin_factor
            dy_fr = results_row['field_fullres_dy'] / bin_factor
            ax.arrow(
                target_x_binned + 240, target_y_binned, dx_fr * scale, dy_fr * scale,
                head_width=35, head_length=25, fc='green', ec='darkgreen', linewidth=3.5,
                label=f'Field (full-res): ({results_row["field_fullres_dx"]:+.2f}, {results_row["field_fullres_dy"]:+.2f}) px',
                zorder=4
            )
        
        title = f"Heat Impulse: {img_label}"
        if img_label == "Test":
            dx_excess_binned = results_row['diff_target_vs_binned_dx']
            dy_excess_binned = results_row['diff_target_vs_binned_dy']
            title += f"\nTarget excess (vs binned): ({dx_excess_binned:+.2f}, {dy_excess_binned:+.2f}) px"
        
        ax.set_title(title, fontsize=13, fontweight='bold')
        ax.set_xlabel('X (binned)', fontsize=11)
        ax.set_ylabel('Y (binned)', fontsize=11)
        ax.legend(fontsize=10, loc='upper right')
        ax.grid(True, alpha=0.2)
        
        # Render to PIL
        buf = io.BytesIO()
        plt.savefig(buf, format='png', dpi=100, bbox_inches='tight')
        buf.seek(0)
        frames.append(Image.open(buf).convert('RGB'))
        buf.close()
        plt.close(fig)
    
    # Create mini loop: baseline → test → baseline → test → baseline
    anim_frames = frames + frames[::-1] + frames
    
    anim_frames[0].save(
        output_file,
        save_all=True,
        append_images=anim_frames[1:],
        duration=600,
        loop=0
    )
    print(f"✓ Saved: {output_file}")
    
    # Display inline
    display(IPImage(filename=str(output_file)))

# Generate and display for each test frame
for idx, test_file in enumerate(TEST_FILES):
    print(f"\n--- Motion Vectors Animation {idx+1}: {test_file} ---")
    test_path = DATA_DIR / test_file
    test_full = load_fits(test_path)
    row = results_df[results_df['frame'] == test_file].iloc[0]
    
    output_file = OUTPUT_DIR / f"05_motion_vectors_all_three_{idx+1}.gif"
    make_motion_vectors_gif_inline(baseline_full, test_full, row, output_file, bin_factor=16)

## 9. Export Results Table (CSV)

In [ ]:
export_df = results_df.copy()
float_cols = export_df.select_dtypes(include='float').columns
export_df[float_cols] = export_df[float_cols].round(4)

csv_path = OUTPUT_DIR / "results_all_three.csv"
export_df.to_csv(csv_path, index=False)

print(f"\n✓ CSV exported: {csv_path}\n")
print(export_df.to_string(index=False))

## 10. Summary and Interpretation

In [ ]:
print("\n" + "="*70)
print("ANALYSIS SUMMARY")
print("="*70)
print(f"\nBaseline FITS: {BASELINE_FILE}")
print(f"Test frames: {', '.join(TEST_FILES)}")
print(f"Target ROI: {TARGET_ROI_SIZE}×{TARGET_ROI_SIZE} px centred on ({TARGET_X_PX}, {TARGET_Y_PX})")
print(f"Whole field analysis:")
print(f"  - Binned: {FIELD_BIN_FACTOR}× before phase correlation")
print(f"  - Full-res: native {baseline_full.shape} detector")

print("\n" + "-"*70)
print("KEY FINDINGS")
print("-"*70)

for idx, (_, row) in enumerate(results_df.iterrows()):
    print(f"\n{row['frame']}:")
    print(f"  Target ROI motion:           ({row['target_dx']:+.3f}, {row['target_dy']:+.3f}) px")
    print(f"  Field motion (binned):       ({row['field_binned_dx']:+.3f}, {row['field_binned_dy']:+.3f}) px")
    print(f"  Field motion (full-res):     ({row['field_fullres_dx']:+.3f}, {row['field_fullres_dy']:+.3f}) px")
    print(f"  Target excess vs binned:     ({row['diff_target_vs_binned_dx']:+.3f}, {row['diff_target_vs_binned_dy']:+.3f}) px")
    print(f"  Target excess vs full-res:   ({row['diff_target_vs_fullres_dx']:+.3f}, {row['diff_target_vs_fullres_dy']:+.3f}) px")
    
    mag_binned = np.sqrt(row['diff_target_vs_binned_dx']**2 + row['diff_target_vs_binned_dy']**2)
    mag_fullres = np.sqrt(row['diff_target_vs_fullres_dx']**2 + row['diff_target_vs_fullres_dy']**2)
    print(f"  Magnitude (binned):          {mag_binned:.3f} px")
    print(f"  Magnitude (full-res):        {mag_fullres:.3f} px")
    
    consistency = np.sqrt(row['consistency_binned_vs_fullres_dx']**2 + row['consistency_binned_vs_fullres_dy']**2)
    print(f"  Consistency (binned vs full-res): {consistency:.3f} px")

print("\n" + "-"*70)
print("INTERPRETATION")
print("-"*70)
print("\nWhat this means:")
print("- Target motion: shift of the bright spot detected by 160×160 px phase correlation")
print("- Field motion (binned): rigid translation of full detector at 8× bin factor")
print("- Field motion (full-res): rigid translation of full detector at native resolution")
print("- Target excess: difference between target and field motion")
print("  → If > 0.5 px, suggests target moved independently of detector plane")
print("  → Could indicate local optical change, PSF evolution, or detector-plane distortion")
print("\nConsistency check:")
print("- Small consistency value (~0.1 px) → binning is safe; rigid motion dominated")
print("- Large consistency value (>0.5 px) → binning loses information; full-res is more trustworthy")

print("\n" + "-"*70)
print("OUTPUT FILES")
print("-"*70)
for fpath in sorted(OUTPUT_DIR.glob("*")):
    if fpath.is_file():
        size_mb = fpath.stat().st_size / 1e6
        print(f"  {fpath.name:40s}  ({size_mb:.2f} MB)")

print(f"\nAll outputs saved to: {OUTPUT_DIR}")